# 03. 텍스트 기반 HS 코드 분류

완제품 사례 데이터(177건)를 이용해 TF-IDF 기반 분류와 유사 사례 검색을 비교한다. 동일 입력 사례는 같은 Fold에 배치하고, 각 Fold의 테스트 사례는 해당 Fold의 학습 데이터만으로 예측한다.

비교 모델: Dummy 기준선, TF-IDF + Logistic Regression, TF-IDF + Linear SVM, TF-IDF 유사 사례 검색(KNN 방식). 해설서 지식베이스를 결합하는 RAG는 다음 단계에서 별도로 추가한다.

In [ ]:
!pip -q install pandas numpy scikit-learn

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from urllib.parse import quote
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, top_k_accuracy_score, classification_report
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

DATA_FILE = '완제품 모델 데이터_사례_전처리 후 - 사례_전처리 완료.csv'
DATA_URL = 'https://raw.githubusercontent.com/lin1118-code/HS-complete-product/main/' + quote(DATA_FILE)
df = pd.read_csv(DATA_URL, encoding='utf-8-sig').fillna('')
INPUT_COLUMNS = ['Product Name', 'Main Ingredients (KR)', 'Main Ingredients (EN)', 'Product Form and Packaging', 'Intended Use']
TARGET_COLUMN = 'HS Code'
for col in INPUT_COLUMNS + [TARGET_COLUMN]:
    df[col] = df[col].astype(str).str.strip()
df = df[df[TARGET_COLUMN].ne('')].reset_index(drop=True)
df['model_text'] = ('[PRODUCT] ' + df['Product Name'] + ' [INGREDIENTS_KR] ' + df['Main Ingredients (KR)'] + ' [INGREDIENTS_EN] ' + df['Main Ingredients (EN)'] + ' [FORM] ' + df['Product Form and Packaging'] + ' [USE] ' + df['Intended Use'])
df['duplicate_group'] = pd.util.hash_pandas_object(df[INPUT_COLUMNS], index=False).astype(str)
print('전체 데이터:', df.shape, '| HS 코드 수:', df[TARGET_COLUMN].nunique())
display(df[INPUT_COLUMNS + [TARGET_COLUMN]].head())

## 1. 평가 대상 클래스와 그룹 점검

기존 기준 실험과 동일하게 사례 수 4건 이상, 서로 다른 입력 그룹 4개 이상인 HS 코드만 주 평가에 포함한다. 제외 클래스도 원본 데이터에는 보존한다.

In [ ]:
class_counts = df[TARGET_COLUMN].value_counts()
eligible_labels = class_counts[class_counts >= 4].index
eval_df = df[df[TARGET_COLUMN].isin(eligible_labels)].copy()
group_counts = eval_df.groupby(TARGET_COLUMN)['duplicate_group'].nunique()
eligible_labels = group_counts[group_counts >= 4].index
eval_df = eval_df[eval_df[TARGET_COLUMN].isin(eligible_labels)].reset_index(drop=True)
print('전체:', len(df), '건 /', df[TARGET_COLUMN].nunique(), '개 코드')
print('주 평가:', len(eval_df), '건 /', eval_df[TARGET_COLUMN].nunique(), '개 코드')
print('별도 보존:', len(df) - len(eval_df), '건')
display(eval_df.groupby(TARGET_COLUMN).agg(count=(TARGET_COLUMN, 'size'), unique_groups=('duplicate_group', 'nunique')).sort_values('count', ascending=False))
if eval_df.empty:
    raise ValueError('평가 가능한 클래스가 없습니다.')

X = eval_df['model_text'].reset_index(drop=True)
y = eval_df[TARGET_COLUMN].reset_index(drop=True)
groups = eval_df['duplicate_group'].reset_index(drop=True)
labels = np.sort(y.unique())
cv = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=42)
splits = list(cv.split(X, y, groups))
fold_audit = []
for fold, (tr, te) in enumerate(splits, 1):
    train_labels, test_labels = set(y.iloc[tr]), set(y.iloc[te])
    fold_audit.append({'fold': fold, 'train_n': len(tr), 'test_n': len(te), 'train_classes': len(train_labels), 'test_classes': len(test_labels), 'test_classes_missing_from_train': sorted(test_labels - train_labels), 'group_overlap': len(set(groups.iloc[tr]) & set(groups.iloc[te]))})
fold_audit_df = pd.DataFrame(fold_audit)
display(fold_audit_df)
if (fold_audit_df['group_overlap'] != 0).any():
    raise ValueError('동일 입력 그룹이 train/test에 동시에 포함되어 데이터 누수가 발생했습니다.')

## 2. TF-IDF 특징 구성

단어 단위 n-gram과 문자 단위 n-gram을 결합한다. 한글 제품 설명과 영문 원재료명, 표기 변형을 함께 반영하기 위한 설정이다. 벡터라이저는 매 Fold의 학습 데이터에만 적합한다.

In [ ]:
def make_vectorizer():
    return FeatureUnion([
        ('word', TfidfVectorizer(analyzer='word', ngram_range=(1, 2), min_df=1, max_features=30000, sublinear_tf=True)),
        ('char', TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), min_df=1, max_features=30000, sublinear_tf=True))
    ])

def metric_row(model_name, fold, y_true, y_pred, y_score=None, score_labels=None):
    row = {'model': model_name, 'fold': fold, 'accuracy': accuracy_score(y_true, y_pred), 'macro_f1': f1_score(y_true, y_pred, labels=labels, average='macro', zero_division=0), 'weighted_f1': f1_score(y_true, y_pred, average='weighted', zero_division=0)}
    if y_score is not None:
        row['top3_accuracy'] = np.mean([true in [score_labels[i] for i in np.argsort(scores)[-min(3, len(score_labels)):][::-1]] for true, scores in zip(y_true, y_score)])
    return row


## 3. Dummy, Logistic Regression, Linear SVM

Macro-F1은 평가 대상 전체 클래스 목록을 고정해 계산한다. 테스트 Fold에 등장하지 않은 클래스도 0점으로 반영하므로 Fold별 클래스 누락을 숨기지 않는다. Top-3는 모델이 산출한 점수 상위 3개 코드 안에 정답이 포함되는 비율이다.

In [ ]:
results = []
oof_rows = []
for fold, (train_idx, test_idx) in enumerate(splits, 1):
    x_train, x_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    dummy = DummyClassifier(strategy='most_frequent').fit(np.zeros((len(train_idx), 1)), y_train)
    dummy_pred = dummy.predict(np.zeros((len(test_idx), 1)))
    results.append(metric_row('Dummy', fold, y_test, dummy_pred))
    vectorizer = make_vectorizer()
    x_train_vec = vectorizer.fit_transform(x_train)
    x_test_vec = vectorizer.transform(x_test)
    models = {
        'TFIDF_LogisticRegression': LogisticRegression(C=1.0, class_weight='balanced', max_iter=3000),
        'TFIDF_LinearSVM': LinearSVC(C=1.0, class_weight='balanced')
    }
    for name, model in models.items():
        model.fit(x_train_vec, y_train)
        pred = model.predict(x_test_vec)
        scores = model.predict_proba(x_test_vec) if hasattr(model, 'predict_proba') else model.decision_function(x_test_vec)
        results.append(metric_row(name, fold, y_test, pred, scores, model.classes_))
        for idx, true, guess in zip(test_idx, y_test, pred):
            oof_rows.append({'model': name, 'row_index': int(idx), 'true_hs': true, 'predicted_hs': guess, 'fold': fold})

result_df = pd.DataFrame(results)
summary_df = result_df.groupby('model').agg({c:['mean','std'] for c in ['accuracy','macro_f1','weighted_f1','top3_accuracy'] if c in result_df.columns})
display(result_df)
display(summary_df)

## 4. TF-IDF 유사 사례 검색

각 테스트 제품에 대해 해당 Fold의 학습 사례만 검색한다. 코사인 유사도 상위 5개 사례의 유사도를 HS 코드별로 합산해 가장 높은 코드를 예측한다. 검색 결과의 Top-3 후보도 함께 평가한다.

In [ ]:
retrieval_rows = []
retrieval_details = []
for fold, (train_idx, test_idx) in enumerate(splits, 1):
    vectorizer = make_vectorizer()
    train_vec = vectorizer.fit_transform(X.iloc[train_idx])
    test_vec = vectorizer.transform(X.iloc[test_idx])
    similarities = cosine_similarity(test_vec, train_vec)
    train_labels = y.iloc[train_idx].to_numpy()
    for local_i, global_i in enumerate(test_idx):
        order = np.argsort(similarities[local_i])[-5:][::-1]
        votes = defaultdict(float)
        for neighbor_i in order:
            votes[train_labels[neighbor_i]] += max(float(similarities[local_i, neighbor_i]), 0)
        ranked = sorted(votes, key=votes.get, reverse=True)
        pred = ranked[0]
        true = y.iloc[global_i]
        retrieval_rows.append({'fold': fold, 'row_index': int(global_i), 'true_hs': true, 'predicted_hs': pred, 'top3_candidates': ranked[:3], 'top1_similarity': float(similarities[local_i, order[0]])})
        retrieval_details.append({'model': 'TFIDF_CosineTop5', 'fold': fold, 'true_hs': true, 'predicted_hs': pred, 'top3_candidates': ranked[:3]})
retrieval_df = pd.DataFrame(retrieval_rows)
retrieval_metrics = []
for fold, part in retrieval_df.groupby('fold'):
    retrieval_metrics.append({'model':'TFIDF_CosineTop5','fold':fold,'accuracy':accuracy_score(part['true_hs'],part['predicted_hs']),'macro_f1':f1_score(part['true_hs'],part['predicted_hs'],labels=labels,average='macro',zero_division=0),'weighted_f1':f1_score(part['true_hs'],part['predicted_hs'],average='weighted',zero_division=0),'top3_accuracy':np.mean([r in c for r,c in zip(part['true_hs'],part['top3_candidates'])])})
retrieval_metric_df = pd.DataFrame(retrieval_metrics)
display(retrieval_metric_df)
display(retrieval_df.head(10))

## 5. 모델 성능 비교와 오분류 확인

각 모델의 Fold 평균과 표준편차를 비교한다. 현재 표본 수가 작고 클래스 불균형이 있으므로 Macro-F1과 Top-3 정확도를 함께 확인한다. 성능 차이가 작다면 단일 수치만으로 우열을 단정하지 않는다.

In [ ]:
all_results = pd.concat([result_df, retrieval_metric_df], ignore_index=True)
comparison = all_results.groupby('model').agg(accuracy_mean=('accuracy','mean'), accuracy_std=('accuracy','std'), macro_f1_mean=('macro_f1','mean'), macro_f1_std=('macro_f1','std'), weighted_f1_mean=('weighted_f1','mean'), top3_accuracy_mean=('top3_accuracy','mean')).sort_values('macro_f1_mean', ascending=False)
display(comparison)
display(retrieval_df[retrieval_df['true_hs'] != retrieval_df['predicted_hs']].head(20))

## 6. 결과 저장

결과는 Colab 세션의 `/content/hs_text_results`에 저장한다. 필요한 파일을 내려받아 GitHub에 별도 업로드할 수 있다.

In [ ]:
OUTPUT_DIR = Path('/content/hs_text_results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
fold_audit_df.to_csv(OUTPUT_DIR / 'text_fold_audit.csv', index=False, encoding='utf-8-sig')
result_df.to_csv(OUTPUT_DIR / 'text_model_fold_results.csv', index=False, encoding='utf-8-sig')
retrieval_df.to_csv(OUTPUT_DIR / 'tfidf_retrieval_predictions.csv', index=False, encoding='utf-8-sig')
retrieval_metric_df.to_csv(OUTPUT_DIR / 'tfidf_retrieval_fold_results.csv', index=False, encoding='utf-8-sig')
comparison.to_csv(OUTPUT_DIR / 'text_model_comparison.csv', encoding='utf-8-sig')
print('저장 완료:', OUTPUT_DIR)
print('저장 파일:', sorted(p.name for p in OUTPUT_DIR.glob('*.csv')))